# Phase 2: Exploratory Spatial Data Analysis (ESDA) for Banking Network Optimization

**Project:** Geospatial Network Optimization for Banking in Kenyan Cities
**Author:** Manus AI
**Date:** March 13, 2026

## Introduction
This Jupyter Notebook serves as the core of Phase 2, focusing on Exploratory Spatial Data Analysis (ESDA). Building upon the robust data ingestion from Phase 1, we will now delve into the spatial relationships between banking infrastructure, population distribution, and mobility patterns. The goal is to identify underserved areas, quantify accessibility, and lay the groundwork for advanced prescriptive analytics. This phase will leverage advanced spatial statistics to uncover hidden patterns and provide data-driven insights for strategic decision-making, demonstrating capabilities essential for a Senior Data Scientist role at Absa Africa.

## Setup and Configuration
First, we set up our Python environment, import necessary libraries, and establish a connection to our PostGIS database. We ensure all paths are compatible with a Windows environment.

In [1]:
import os
import geopandas as gpd
import pandas as pd
from sqlalchemy import create_engine
import urllib.parse
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium.plugins import HeatMap
import contextily as ctx
import osmnx as ox
import networkx as nx
import libpysal as lps
from esda.moran import Moran, Moran_Local
from pysal.lib import weights
from mgwr.gwr import GWR
from mgwr.sel_bw import Sel_BW
import numpy as np

# Database connection details
DB_USER = "geospatial"
DB_PASSWORD = "RennySweetpea@1" # IMPORTANT: Replace with your actual password
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "geospatial_banking_kenya"

# Encode password for SQLAlchemy connection string
safe_password = urllib.parse.quote_plus(DB_PASSWORD)
db_connection_str = f"postgresql://{DB_USER}:{safe_password}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(db_connection_str)

# Define the base directory for your project
BASE_DIR = r"C:\Users\Jones Mbela\Desktop\RENNY\AI AND ML\Geospatial Network Optimization"

AOI_DIR = os.path.join(BASE_DIR,"Data","aoi_data")

print("Environment setup complete. Database connection established.")

Environment setup complete. Database connection established.


## 1. Data Loading and Initial Exploration
We load the necessary geospatial data for Nairobi, Mombasa, and Kisumu from our PostGIS database. This includes administrative boundaries, banking Points of Interest (POIs), population data, and Facebook Mobility data.

In [5]:
cities = ['nairobi', 'mombasa', 'kisumu']

data = {}
for city in cities:
    print(f"Loading data for {city.title()}...")
    try:
        # Load AOI
        data[city] = {}
        data[city]['aoi'] = gpd.read_file(os.path.join(AOI_DIR, f"{city}_aoi.geojson"))
        data[city]['aoi'] = data[city]['aoi'].to_crs(epsg=4326)

        # Load POIs
        data[city]['pois'] = gpd.read_postgis(f"SELECT * FROM {city}_osm_pois", engine, geom_col='geometry')
        data[city]['pois'] = data[city]['pois'].to_crs(epsg=4326)

        # Load Population Points
        data[city]['population'] = gpd.read_postgis(f"SELECT * FROM {city}_worldpop_points", engine, geom_col='geometry')
        data[city]['population'] = data[city]['population'].to_crs(epsg=4326)

        # Load Facebook Mobility Data
        try:
            data[city]['mobility'] = gpd.read_postgis(f"SELECT * FROM {city}_fb_mobility", engine, geom_col='geometry')
            data[city]['mobility'] = data[city]['mobility'].to_crs(epsg=4326)
        except Exception as e:
            print(f"  Warning: No Facebook Mobility data found for {city.title()} or table does not exist.")
            data[city]['mobility'] = gpd.GeoDataFrame()

        print(f"  {city.title()} data loaded successfully.")
    except Exception as e:
        print(f"  Error loading data for {city.title()}: {e}")

print("\nData loading complete.")

Loading data for Nairobi...
  Nairobi data loaded successfully.
Loading data for Mombasa...
  Mombasa data loaded successfully.
Loading data for Kisumu...
  Kisumu data loaded successfully.

Data loading complete.


## 2. Spatial Overlay Analysis: Identifying Underserved Areas
In this section, we create interactive maps that combine population heatmaps with banking POIs using Folium

In [12]:
for city in cities:
    print(f"Generating spatial overlay map for {city.title()}...")
    center_lat = data[city]["aoi"].geometry.centroid.y.mean()
    center_lon = data[city]["aoi"].geometry.centroid.x.mean()
    m = folium.Map(location=[center_lat, center_lon], zoom_start=11, tiles="cartodbpositron")

    if not data[city]["population"].empty:
        population_data = data[city]["population"].copy()
        population_data["lat"] = population_data.geometry.y
        population_data["lon"] = population_data.geometry.x
        heat_data = [[row["lat"], row["lon"], row["population"]] for index, row in population_data.iterrows()]
        HeatMap(heat_data, radius=15).add_to(m)

    if not data[city]["pois"].empty:
        pois_df = data[city]["pois"]
        for idx, row in pois_df.iterrows():
            centroid = row.geometry.centroid
            poi_type = row.get('shop') or row.get('amenity') or row.get('brand') or 'Point of Interest'
            poi_name = row.get('name') or 'Unamed Location'
            folium.Marker(
                location=[centroid.y, centroid.x],
                popup=folium.Popup(f"<b>{poi_name}</b><br>Type: {poi_type}", max_width=300),
                icon=folium.Icon(color="blue", icon='info-sign')
            ).add_to(m)

    folium.GeoJson(
        data[city]["aoi"].geometry,
        name= "Area of Interest",
        style_function= lambda x:{'color': 'black', 'fillColor': 'transparent'}
    ).add_to(m)
    map_path = os.path.join(os.getcwd(), f"{city}_population_pois_overlay.html")
    m.save(map_path)
    print(f"  Map saved to {map_path}")

Generating spatial overlay map for Nairobi...


C:\Users\Jones Mbela\AppData\Local\Temp\ipykernel_28832\4095422871.py:3: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center_lat = data[city]["aoi"].geometry.centroid.y.mean()
C:\Users\Jones Mbela\AppData\Local\Temp\ipykernel_28832\4095422871.py:4: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center_lon = data[city]["aoi"].geometry.centroid.x.mean()


  Map saved to c:\Users\Jones Mbela\Desktop\RENNY\AI AND ML\Geospatial Network Optimization\Notebooks\nairobi_population_pois_overlay.html
Generating spatial overlay map for Mombasa...


C:\Users\Jones Mbela\AppData\Local\Temp\ipykernel_28832\4095422871.py:3: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center_lat = data[city]["aoi"].geometry.centroid.y.mean()
C:\Users\Jones Mbela\AppData\Local\Temp\ipykernel_28832\4095422871.py:4: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center_lon = data[city]["aoi"].geometry.centroid.x.mean()


  Map saved to c:\Users\Jones Mbela\Desktop\RENNY\AI AND ML\Geospatial Network Optimization\Notebooks\mombasa_population_pois_overlay.html
Generating spatial overlay map for Kisumu...


C:\Users\Jones Mbela\AppData\Local\Temp\ipykernel_28832\4095422871.py:3: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center_lat = data[city]["aoi"].geometry.centroid.y.mean()
C:\Users\Jones Mbela\AppData\Local\Temp\ipykernel_28832\4095422871.py:4: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center_lon = data[city]["aoi"].geometry.centroid.x.mean()


  Map saved to c:\Users\Jones Mbela\Desktop\RENNY\AI AND ML\Geospatial Network Optimization\Notebooks\kisumu_population_pois_overlay.html


In [8]:
print(data[city]["pois"].columns)

Index(['element', 'id', 'geometry', 'addr:housenumber', 'name', 'operator',
       'shop', 'addr:city', 'addr:street', 'brand',
       ...
       'ele', 'roof:levels', 'roof:shape', 'nohousenumber', 'brand:ar',
       'name:ar', 'landuse', 'building:levels:underground',
       'building:material', 'type'],
      dtype='object', length=123)


## 3. Distance-to-Service Analysis: Quantifying Accessibility
This section calculates network distances to the nearest banking POI.

In [ ]:
def calculate_network_distance_to_nearest_poi(city_name, population_gdf, pois_gdf, network_type="drive"):
    print(f"  Calculating network distances for {city_name}...")
    if population_gdf.empty or pois_gdf.empty:
        return population_gdf

    G = ox.graph_from_place(f"{city_name}, Kenya", network_type=network_type)
    G_proj = ox.project_graph(G)
    G_proj = ox.add_edge_speeds(G_proj)
    G_proj = ox.add_edge_travel_times(G_proj)

    pois_proj = pois_gdf.to_crs(G_proj.graph["crs"])
    population_proj = population_gdf.to_crs(G_proj.graph["crs"])

    orig_nodes = ox.nearest_nodes(G_proj, population_proj.geometry.x, population_proj.geometry.y)
    pois_proj = pois_gdf.to_crs(G_proj.graph["crs"])

    target_nodes = ox.nearest_nodes(
        G_proj,
        X=pois_proj.geometry.centroid.x,
        Y=pois_proj.geometry.centroid.y
    )

    distances = []
    for i, orig in enumerate(orig_nodes):
        min_dist = np.inf
        for target in target_nodes:
            try:
                dist = nx.shortest_path_length(G_proj, orig, target, weight="length")
                if dist < min_dist:
                    min_dist = dist
            except nx.NetworkXNoPath:
                continue
        distances.append(min_dist if min_dist != np.inf else np.nan)

    population_gdf["distance_to_nearest_poi"] = distances
    return population_gdf

for city in cities:
    data[city]["population"] = calculate_network_distance_to_nearest_poi(city.title(), data[city]["population"], data[city]["pois"])

  Calculating network distances for Nairobi...


## 4. Statistical Spatial Analysis: Uncovering Patterns
Calculating Moran's I for spatial autocorrelation.

In [ ]:
for city in cities:
    print(f"Calculating Moran's I for {city.title()}...")
    pop_with_dist = data[city]["population"].dropna(subset=["distance_to_nearest_poi"])
    if len(pop_with_dist) < 10:
        continue

    wq = weights.KNN.from_dataframe(pop_with_dist, k=8)
    wq.transform = "R"
    y = pop_with_dist["distance_to_nearest_poi"]
    moran = Moran(y, wq)
    print(f"  Global Moran's I: {moran.I:.4f}, p-value: {moran.p_sim:.4f}")

## 5. Conclusion and Next Steps
Phase 2 complete. Ready for Phase 3 Optimization.